# Version 24 — Gain-scheduled / LPV HVAC control

**Hypothesis:** Regime-specific zone controllers can generalize better across weather and thermal operating conditions than one fixed set of feedback gains.

A *linear parameter-varying (LPV-inspired)* PI-D regulator interpolates cooling and heating gains independently over measured outside-air temperature at **280 K (cold), 294 K (mild), and 308 K (hot)**. The three gain sets are engineering defaults, not fitted or stability-certified LPV designs. It uses **continuous convex interpolation** to avoid gain jumps at regime boundaries; AHU on/off switching is not hysteretic.

For each zone, demand is based on the excursion *outside* its current heating/cooling setpoint band, plus a filtered five-minute temperature trend and a bounded, anti-windup integral. Demand is mutually exclusive between heating and cooling and zero in the comfort deadband. State is cleared when setpoints change or sensors jump implausibly; demands without usable dampers/reheat valves are ignored. The physical AHU/VAV/economizer/hot-water dispatcher is retained from V21 to isolate the feedback hypothesis. All commands are bounded and indexed by episode metadata.

**Controls / ablation:** Set `TransferPolicy.ENABLE_GAIN_SCHEDULING = False` to use the **mild-weather gains for every outdoor condition**, without changing physical dispatch. Missing or invalid outdoor data also falls back to mild gains. This is a controlled fixed-gain comparison, not a temperature-independent physical plant.

**Validation:** Run `python "Co_Kaggle/g8/submissions/test_version24.py"` for structural, interpolation, gain-ablation, thermal-response, reset, saturation and Kaggle action-contract checks. Run Kaggle with the notebook's final official inference-server cell. To test generalization, compare paired rollouts across held-out cold/mild/hot weather, occupied/eco windows, buildings and season shifts, reporting official score, comfort tails, energy, equipment switching and score variance. No performance or stability improvement is claimed before official simulator evaluation.


In [ ]:
"""Version 24: outdoor-temperature gain-scheduled / LPV HVAC feedback.

Each zone runs a bounded PI-D regulator with mode-specific gains.
Cooling and heating gains are linearly interpolated between three measured
outdoor-temperature operating points (cold/mild/hot), without hard switches.
The existing thermodynamic HVAC dispatch maps demand to physical actuators.
Runtime is O(number of AHUs + number of zones) per decision step.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Gain-scheduled zone PI-D feedback with shared physical HVAC dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    Metadata provides all equipment topology and sensor indices.
    """

    # Plant command limits retained from the tested physical dispatch.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    MIN_VAV_FRACTION = 0.10
    MIN_CONTROL_DEMAND = 0.05
    MIN_FAN_PRESSURE_PA = 200.0
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15
    COOLING_HEADROOM_GAIN_K = 8.0
    COOLING_DAMPER_GAIN = 0.72
    HEATING_DAMPER_GAIN = 0.50
    HEATING_VALVE_GAIN = 0.74

    # Piecewise-affine LPV gain schedule, in Kelvin; values are
    # (proportional [1/K], integral [1/(K*step)], derivative [step/K]).
    # These engineering initializations require simulator calibration.
    OUTDOOR_KNOTS_K = (280.0, 294.0, 308.0)
    COOLING_GAINS = (
        (0.40, 0.020, 0.65),  # Cold weather: natural cooling is effective.
        (0.65, 0.035, 0.90),  # Mild weather.
        (0.96, 0.050, 1.20),  # Hot weather: require stronger active cooling.
    )
    HEATING_GAINS = (
        (1.00, 0.050, 1.00),  # Cold weather: stronger reheat feedback.
        (0.68, 0.035, 0.85),  # Mild weather.
        (0.42, 0.020, 0.65),  # Hot weather: less heating load.
    )
    ENABLE_GAIN_SCHEDULING = True  # False fixes both controllers at mild gains.
    RATE_FILTER_ALPHA = 0.55
    MAX_VALID_TEMPERATURE_CHANGE_K = 1.5
    MAX_INTEGRAL_ERROR_K_STEPS = 6.0
    INTEGRAL_MEMORY = 0.90

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        outdoor_stats = meta.get(
            "observation_normalization_metadata", {}
        ).get("outside_air_temperature_sensor", {})
        self.outdoor_mean = float(
            outdoor_stats.get("sample_mean", float("nan"))
        )
        self.outdoor_std = float(
            outdoor_stats.get("sample_std", float("nan"))
        )

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Clear PI-D integrators and filtered rates between episodes."""
        self._state = {}

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _outside_temperature(self, obs, index):
        """Return outdoor temperature in Kelvin, or NaN if unavailable."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")

        has_normalization = (
            np.isfinite(self.outdoor_mean)
            and np.isfinite(self.outdoor_std)
            and self.outdoor_std > 0
        )
        kelvin = (
            raw * self.outdoor_std + self.outdoor_mean
            if has_normalization else raw
        )
        return float(kelvin) if 210.0 <= kelvin <= 330.0 else float("nan")

    def _scheduled_gains(self, outside_k, heating=False):
        """Interpolate gains over the measured outdoor operating point.

        Convex interpolation makes the controller continuous at regime
        boundaries. Missing or corrupt weather data uses the mild gains.
        The ablation switch removes scheduling but changes nothing else.
        """
        table = self.HEATING_GAINS if heating else self.COOLING_GAINS
        if not self.ENABLE_GAIN_SCHEDULING or not np.isfinite(outside_k):
            return table[1]

        knots = self.OUTDOOR_KNOTS_K
        if outside_k <= knots[0]:
            return table[0]
        if outside_k >= knots[-1]:
            return table[-1]

        segment = 0 if outside_k < knots[1] else 1
        mix = (outside_k - knots[segment]) / (
            knots[segment + 1] - knots[segment]
        )
        return tuple(
            (1.0 - mix) * first + mix * second
            for first, second in zip(table[segment], table[segment + 1])
        )

    def _gain_scheduled_status(self, status, outside_k):
        """Compute signed zone demand using a gain-scheduled PI-D regulator.

        The controller measures deviation *outside* the comfort band. A
        positive temperature trend increases cooling effort, and a negative
        trend increases heating effort. Integrator history is invalidated on
        mode/setpoint changes or implausible sensor jumps. The temperature
        derivative is filtered, but equipment run/on-off commands themselves
        are not hysteretic.
        """
        zone = status["actions"]
        key = zone["temperature"]
        temperature = status["temp"]
        low, high = status["low"], status["high"]

        # State tuple: temperature, filtered_rate, integral, mode, low, high.
        previous = self._state.get(key)
        rate, previous_integral, previous_mode = 0.0, 0.0, 0
        if previous is not None:
            previous_temp, previous_rate, previous_integral, previous_mode, old_low, old_high = previous
            delta = temperature - previous_temp
            if abs(delta) <= self.MAX_VALID_TEMPERATURE_CHANGE_K:
                alpha = self.RATE_FILTER_ALPHA
                rate = alpha * delta + (1.0 - alpha) * previous_rate
            else:
                # A sensor discontinuity invalidates the controller's state.
                previous_integral, previous_mode = 0.0, 0

            if (low, high) != (old_low, old_high):
                previous_integral, previous_mode = 0.0, 0

        if temperature > high:
            mode = 1  # Cooling
            error = temperature - high
            unfavorable_rate = max(rate, 0.0)
            heating = False
        elif temperature < low:
            mode = -1  # Heating
            error = low - temperature
            unfavorable_rate = max(-rate, 0.0)
            heating = True
        else:
            mode = 0  # Within deadband: stop and reset integration.
            error = 0.0
            unfavorable_rate = 0.0
            heating = False

        integral, demand = 0.0, 0.0
        if mode:
            if mode != previous_mode:
                previous_integral = 0.0

            kp, ki, kd = self._scheduled_gains(outside_k, heating=heating)
            integral = min(
                self.MAX_INTEGRAL_ERROR_K_STEPS,
                self.INTEGRAL_MEMORY * previous_integral + error,
            )
            demand = kp * error + ki * integral + kd * unfavorable_rate
            if demand > 1.0:
                # Conditional anti-windup: do not accumulate while saturated.
                integral = min(
                    self.MAX_INTEGRAL_ERROR_K_STEPS,
                    self.INTEGRAL_MEMORY * previous_integral,
                )
                demand = kp * error + ki * integral + kd * unfavorable_rate
            demand = self._clamp(demand, 0.0, 1.0)

        self._state[key] = (temperature, rate, integral, mode, low, high)
        status["lpv_cooling"] = demand if mode == 1 else 0.0
        status["lpv_heating"] = demand if mode == -1 else 0.0
        return status

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "low": low,
            "high": high,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _actionable_demands(self, status):
        """Ignore requests a zone cannot physically satisfy.

        Cooling requires a VAV damper. Heating requires both airflow and a
        reheat valve. Deadband-sized demands must not dispatch equipment,
        including when another zone on the same AHU is active.
        """
        zone = status["actions"]
        if zone["damper"] is None:
            return 0.0, 0.0

        cooling = status["lpv_cooling"]
        heating = status["lpv_heating"] if zone["valve"] is not None else 0.0
        if cooling < self.MIN_CONTROL_DEMAND:
            cooling = 0.0
        if heating < self.MIN_CONTROL_DEMAND:
            heating = 0.0
        return cooling, heating

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        cooling_zones = [
            zone for zone in zones
            if self._actionable_demands(zone)[0] > 0.0
        ]
        hottest = max((zone["lpv_cooling"] for zone in cooling_zones), default=0.0)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if not cooling_zones:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            mean_hot_k = float(np.mean([zone["temp"] for zone in cooling_zones]))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (cooling_zones
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(
        self, statuses, delivered_air_k, hot_water_demands, continuous
    ):
        """Apply VAV airflow and reheat demands with physical headroom.

        Cooling airflow scales inversely with temperature headroom; heating
        uses moderate airflow and local reheat. Subthreshold or impossible
        requests remain at the minimum damper opening.
        """
        total_opening = 0.0
        peak_opening = 0.0

        for status in statuses:
            zone = status["actions"]
            cooling, heating = self._actionable_demands(status)

            if cooling > 0.0:
                headroom_k = max(2.5, status["temp"] - delivered_air_k)
                airflow_gain = self._clamp(
                    self.COOLING_HEADROOM_GAIN_K / headroom_k, 0.45, 1.6
                )
                damper = self._clamp(
                    0.12 + self.COOLING_DAMPER_GAIN * cooling * airflow_gain,
                    self.MIN_VAV_FRACTION, 0.98
                )
            elif heating > 0.0:
                damper = self._clamp(
                    0.22 + self.HEATING_DAMPER_GAIN * heating, 0.18, 0.88
                )
                if zone["valve"] is not None:
                    valve = self._clamp(
                        0.12 + self.HEATING_VALVE_GAIN * heating, 0.0, 1.0
                    )
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append(
                        (max(status["cold"], 0.1 * heating), valve)
                    )
            else:
                damper = self.MIN_VAV_FRACTION

            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
            total_opening += damper
            peak_opening = max(peak_opening, damper)

        return total_opening / len(statuses), peak_opening

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Dispatch an AHU only for actionable zone heating or cooling."""
        if not statuses:
            return
        worst_demand = max(
            max(self._actionable_demands(zone)) for zone in statuses
        )
        if worst_demand == 0.0:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Run one controller step: zones -> gain-scheduled feedback -> AHU -> water.

        State is per-zone temperature, filtered rate, and bounded integrators.
        Equipment commands are recomputed every step; no online fitting.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor):
                outdoors.append(outdoor)

            statuses = []
            for zone in ahu["zones"]:
                status = self._read_zone(obs, zone, fallback)
                if status is None:
                    # Reset the zone controller after any invalid reading.
                    self._state.pop(zone["temperature"], None)
                    continue
                statuses.append(self._gain_scheduled_status(status, outdoor))
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}



In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
